In [5]:
import pandas as pd
import numpy as np

# Identifying Duplicated Species

In [6]:
orthoDist = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.parquet")
display(orthoDist)

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score,EuclidDist,PearDist,TEC,EuclidDistNorm,PearDistNorm
Human ID,,,,,,,,,
ENSG00000000003,ENSG00000000003,ENSMUSG00000067377,ortholog_one2one,100.0,44.054842,1.130272,0.0000,0.044055,1.130272
ENSG00000000005,ENSG00000000005,ENSMUSG00000031250,ortholog_one2one,100.0,5.172072,1.141118,0.1250,0.005172,1.141118
ENSG00000000419,ENSG00000000419,ENSMUSG00000078919,ortholog_one2one,100.0,188.287732,1.211158,0.0000,0.188288,1.211158
ENSG00000000457,ENSG00000000457,ENSMUSG00000026584,ortholog_one2one,100.0,37.114733,0.814870,0.0000,0.037115,0.814870
ENSG00000000460,ENSG00000000460,ENSMUSG00000041406,ortholog_one2one,0.0,9.113600,0.959842,0.1875,0.009114,0.959842
...,...,...,...,...,...,...,...,...,...
NaN,ENSG00000310576,ENSMUSG00000035595,ortholog_one2one,100.0,NaN,NaN,NaN,NaN,NaN
NaN,ENSG00000310579,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NaN,ENSG00000310583,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# Identifies the number of mouse genes associated with each human gene.
groupedHumanGene = orthoDist.groupby("Gene stable ID")["Mouse gene stable ID"].apply(list).reset_index(name="Mouse gene stable ID")

# Adds in the homology type to the dataframe.
groupedHumanGene = groupedHumanGene.merge(orthoDist.loc[:, ["Gene stable ID", "Mouse homology type"]])

# Identifies the human genes that have more than one mouse gene associated with them.
groupedHumanGene["Num Mouse Dupes"] = groupedHumanGene["Mouse gene stable ID"].apply(len)

# Creates a new column that identifies which one-to-many gene experienced a duplication event in humans. 
groupedHumanGene["Duplicated Species"] = np.where((groupedHumanGene["Gene stable ID"].isin(groupedHumanGene[groupedHumanGene["Num Mouse Dupes"] > 1]["Gene stable ID"])) & (groupedHumanGene["Mouse homology type"] == "ortholog_one2many"), "Human", "NA")

In [8]:
groupedHumanGene

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Num Mouse Dupes,Duplicated Species
0,ENSG00000000003,[ENSMUSG00000067377],ortholog_one2one,1,NA
1,ENSG00000000005,[ENSMUSG00000031250],ortholog_one2one,1,NA
2,ENSG00000000419,[ENSMUSG00000078919],ortholog_one2one,1,NA
3,ENSG00000000457,[ENSMUSG00000026584],ortholog_one2one,1,NA
4,ENSG00000000460,[ENSMUSG00000041406],ortholog_one2one,1,NA
...,...,...,...,...,...
28009,ENSG00000310576,[ENSMUSG00000035595],ortholog_one2one,1,NA
28010,ENSG00000310579,[nan],NaN,1,NA
28011,ENSG00000310583,[nan],NaN,1,NA
28012,ENSG00000310590,[nan],NaN,1,NA


In [9]:
# Transfers the information above to the orthologTableDist dataframe.
orthoDistHumanDup = orthoDist.merge(groupedHumanGene.loc[:, ["Gene stable ID", "Duplicated Species", "Num Mouse Dupes"]], left_on="Gene stable ID", right_on="Gene stable ID", how="outer")

In [10]:
# Same steps as three cells above, but for mouse genes.
groupedMouseGene = orthoDist.groupby("Mouse gene stable ID")["Gene stable ID"].apply(list).reset_index(name="Gene stable ID")
groupedMouseGene = groupedMouseGene.merge(orthoDist.loc[:, ["Mouse gene stable ID", "Mouse homology type"]])
groupedMouseGene["Num Human Dupes"] = groupedMouseGene["Gene stable ID"].apply(len)
groupedMouseGene["Duplicated Species"] = np.where((groupedMouseGene["Mouse gene stable ID"].isin(groupedMouseGene[groupedMouseGene["Num Human Dupes"] > 1]["Mouse gene stable ID"])) & (groupedMouseGene["Mouse homology type"] == "ortholog_one2many"), "Mouse", "NA")

In [11]:
groupedMouseGene

,Mouse gene stable ID,Gene stable ID,Mouse homology type,Num Human Dupes,Duplicated Species
0,ENSMUSG00000000001,[ENSG00000065135],ortholog_one2one,1,NA
1,ENSMUSG00000000028,[ENSG00000093009],ortholog_one2one,1,NA
2,ENSMUSG00000000037,[ENSG00000102098],ortholog_one2one,1,NA
3,ENSMUSG00000000049,[ENSG00000091583],ortholog_one2one,1,NA
4,ENSMUSG00000000056,[ENSG00000141562],ortholog_one2one,1,NA
...,...,...,...,...,...
22464,ENSMUSG00000144248,[ENSG00000289360],ortholog_one2one,1,NA
22465,ENSMUSG00000144259,[ENSG00000288706],ortholog_one2one,1,NA
22466,ENSMUSG00000144287,[ENSG00000255154],ortholog_one2one,1,NA
22467,ENSMUSG00001074846,[ENSG00000229972],ortholog_one2one,1,NA


In [12]:
# Transfers the information above to the orthologDistTable dataframe.
orthoDistHumanMouseDup = orthoDistHumanDup.merge(groupedMouseGene.loc[:, ["Mouse gene stable ID", "Duplicated Species", "Num Human Dupes"]], on="Mouse gene stable ID", how="outer")

# Because there are two separate "Duplicated Species" columns, I am going to combine their information into one.
# If the human "Duplicated Species" column is "NA" then we use the mouse "Duplicated Species" column. It will either be "Mouse" or stay "NA."
orthoDistHumanMouseDup["Duplicated Species"] = np.where(orthoDistHumanMouseDup["Duplicated Species_x"] == "NA", orthoDistHumanMouseDup["Duplicated Species_y"], orthoDistHumanMouseDup["Duplicated Species_x"])

# Drop the two separate "Duplicated Species" columns.
orthoDistHumanMouseDup = orthoDistHumanMouseDup.loc[:, ~orthoDistHumanMouseDup.columns.isin(["Duplicated Species_x", "Duplicated Species_y"])].drop_duplicates()
display(orthoDistHumanMouseDup)

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score,EuclidDist,PearDist,TEC,EuclidDistNorm,PearDistNorm,Num Mouse Dupes,Num Human Dupes,Duplicated Species
0,ENSG00000065135,ENSMUSG00000000001,ortholog_one2one,100.0,197.746445,0.151442,0.0000,0.197746,0.151442,1,1.0,NA
1,ENSG00000093009,ENSMUSG00000000028,ortholog_one2one,100.0,24.533616,1.001638,0.1875,0.024534,1.001638,1,1.0,NA
2,ENSG00000102098,ENSMUSG00000000037,ortholog_one2one,100.0,2.084639,0.161786,0.0625,0.002085,0.161786,1,1.0,NA
3,ENSG00000091583,ENSMUSG00000000049,ortholog_one2one,100.0,1776.546401,0.000003,0.2500,1.776547,0.000003,1,1.0,NA
4,ENSG00000141562,ENSMUSG00000000056,ortholog_one2one,100.0,70.950296,0.661562,0.0000,0.070950,0.661562,1,1.0,NA
...,...,...,...,...,...,...,...,...,...,...,...,...
3884923,ENSG00000310562,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN
3884924,ENSG00000310579,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN
3884925,ENSG00000310583,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN
3884926,ENSG00000310590,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN


In [14]:
orthoDistHumanMouseDup.to_csv("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupe.csv", index=False)
orthoDistHumanMouseDup.to_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupe.parquet", index=False)